# Study 825 — Oil Predicts Equities — the teardown

The predictive-regression slope with a Newey-West HAC *t*, the R², the tercile cross-check, the 2,000-permutation placebo, the two-era robustness cut, the costed monthly timer (vs a buy-&-hold benchmark), and the 20-seed synthetic control.

In [1]:
R = {'start': '2006-05-31', 'end': '2026-05-31', 'n_months': 241, 'oil': 'USO', 'eq': 'SPY', 'fingerprint': 'effe48bc1a4f', 'beta': 0.0136, 't_nw': 0.33, 't_ols': 0.52, 'r2_pct': 0.11, 'alpha_pct': 0.99, 'fwd_down_pct': 0.95, 'fwd_up_pct': 0.83, 'welch_t': -0.15, 'placebo_obs': 0.0136, 'placebo_mean': 0.0002, 'placebo_sd': 0.026, 'placebo_p': 0.587, 'placebo_draws': 2000, 'era1_n': 116, 'era1_beta': 0.0689, 'era1_t': 1.23, 'era1_r2': 2.15, 'era2_n': 125, 'era2_beta': -0.0234, 'era2_t': -0.5, 'era2_r2': 0.41, 'ls1_gross': 0.2, 'ls1_net': 0.17, 'ls1_t': 0.58, 'ls1_sharpe': 0.13, 'ls1_ann': 2.0, 'ls5_net': 0.132, 'ls5_t': 0.45, 'hit': 0.506, 'lf1_net': 0.593, 'lf1_t': 2.6, 'lf1_sharpe': 0.58, 'lf1_ann': 7.1, 'spy_bh': 0.973, 'spy_bh_t': 3.44, 'spy_bh_sharpe': 0.77, 'null_mean_t': 0.01, 'null_sd_t': 1.03, 'null_fire': 1, 'planted_beta': -0.1951, 'planted_t': -5.91, 'planted_r2': 12.4}

## The headline — predictive regression  `r_equity[t+1] = a + b·r_oil[t]`

Monthly, one documented lag: oil return known at the close of month `t`, equity return realised over month `t+1`. Slope `b` is the Driesprong coefficient.

In [2]:
print(f"slope beta   : {R['beta']:+.4f}   NW(6) t = {R['t_nw']:+.2f}   "
      f"OLS t = {R['t_ols']:+.2f}   R2 = {R['r2_pct']:.2f}%")
print(f"alpha        : {R['alpha_pct']:+.2f}%/mo   n = {R['n_months']} months")
print(f"tercile check: fwd SPY after oil-down {R['fwd_down_pct']:+.2f}% vs "
      f"after oil-up {R['fwd_up_pct']:+.2f}% (Welch t = {R['welch_t']:+.2f})")
print('  claim: beta < 0 (oil up -> stocks down next month). Found: beta ~ 0, wrong sign.')

slope beta   : +0.0136   NW(6) t = +0.33   OLS t = +0.52   R2 = 0.11%
alpha        : +0.99%/mo   n = 241 months
tercile check: fwd SPY after oil-down +0.95% vs after oil-up +0.83% (Welch t = -0.15)
  claim: beta < 0 (oil up -> stocks down next month). Found: beta ~ 0, wrong sign.


## Placebo — permute the target, keep the predictor (2,000 draws)

In [3]:
print(f"observed beta {R['placebo_obs']:+.4f} vs placebo mean {R['placebo_mean']:+.4f} "
      f"(sd {R['placebo_sd']:.4f}) -> two-sided p = {R['placebo_p']:.3f}")

observed beta +0.0136 vs placebo mean +0.0002 (sd 0.0260) -> two-sided p = 0.587


## Robustness — two eras (split 2016-01-01)

In [4]:
print(f"2006-2015 (n={R['era1_n']}): beta {R['era1_beta']:+.4f}  NW t = {R['era1_t']:+.2f}  R2 = {R['era1_r2']:.2f}%")
print(f"2016-2026 (n={R['era2_n']}): beta {R['era2_beta']:+.4f}  NW t = {R['era2_t']:+.2f}  R2 = {R['era2_r2']:.2f}%")
print('  the slope even flips sign across eras -- no stable relation.')

2006-2015 (n=116): beta +0.0689  NW t = +1.23  R2 = 2.15%
2016-2026 (n=125): beta -0.0234  NW t = -0.50  R2 = 0.41%
  the slope even flips sign across eras -- no stable relation.


## The timer — can you get paid for it?

Trade `-sign(oil_ret[t])` of SPY next month; one-way cost × NAV per rebalance leg, 50 bps/yr borrow on shorts. Compared against just buying and holding SPY.

In [5]:
print(f"long/short 1bp: gross {R['ls1_gross']:+.3f}%/mo -> net {R['ls1_net']:+.3f}%/mo "
      f"(t={R['ls1_t']:+.2f}, Sharpe {R['ls1_sharpe']:.2f}, ~{R['ls1_ann']:+.1f}%/yr, hit {R['hit']:.3f})")
print(f"long/short 5bp: net {R['ls5_net']:+.3f}%/mo (t={R['ls5_t']:+.2f})")
print(f"long/flat  1bp: net {R['lf1_net']:+.3f}%/mo (t={R['lf1_t']:+.2f}, Sharpe {R['lf1_sharpe']:.2f})")
print(f"SPY buy&hold  : {R['spy_bh']:+.3f}%/mo (t={R['spy_bh_t']:+.2f}, Sharpe {R['spy_bh_sharpe']:.2f})")
print('  the long/flat timer only \'works\' by inheriting the equity premium -- and still LOSES to buy&hold.')

long/short 1bp: gross +0.200%/mo -> net +0.170%/mo (t=+0.58, Sharpe 0.13, ~+2.0%/yr, hit 0.506)
long/short 5bp: net +0.132%/mo (t=+0.45)
long/flat  1bp: net +0.593%/mo (t=+2.60, Sharpe 0.58)
SPY buy&hold  : +0.973%/mo (t=+3.44, Sharpe 0.77)
  the long/flat timer only 'works' by inheriting the equity premium -- and still LOSES to buy&hold.


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted negative slope.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from oil_equities import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_series(edge=0.0, seed=825+s))['t_nw'] for s in range(20)])
print(f"null (edge=0), 20 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), "
      f"|t|>=2 in {(abs(null_t)>=2).sum()}/20")
planted = st.synthetic_detect(data.synthetic_series(edge=0.35, seed=825))
print(f"planted (edge=0.35): beta = {planted['beta']:+.4f}, NW t = {planted['t_nw']:+.2f}, R2 = {planted['r2_pct']:.2f}%")

null (edge=0), 20 seeds: NW t mean +0.01 (sd 1.03), |t|>=2 in 1/20


planted (edge=0.35): beta = -0.1951, NW t = -5.91, R2 = 12.40%


## Verdict

- **Signal — None.** The Driesprong negative oil→equity predictive slope does **not** replicate on 2006–2026 US ETFs: the slope is **+0.0136** (NW *t* = **+0.33**, R² = **0.11%**) — a flat, wrong-signed nothing. It is p = 0.59 in a 2,000-draw placebo and flips sign across eras (*t* = +1.23 / -0.50). The 20-seed synthetic control recovers a *planted* negative slope cleanly (*t* = -5.91, fires on 1/20 nulls), so the flat real-tape result is a genuine null, not a bug.
- **Tradability — Mirage.** The long/short timer is a coin-flip (net +0.170%/mo, *t* = +0.58, hit 0.506); the long/flat variant looks positive (*t* = +2.60) only because it inherits the equity premium — and it still **loses to buying and holding SPY** (+0.973%/mo, Sharpe 0.77 vs 0.58). No paycheck here.